# Prior-vs-noise toy experiments

Compares drift effects for natural/prior-like sequences vs random Gaussian noise sequences.


In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from utls.toy_experiments import make_toy_experiment_list
from utls.runners_v2 import run_experiment_scores_prior
from utls.sigma_fitting import _compute_auroc_upper_envelope
from utls.analysis_helpers import auroc_to_dprime



In [ ]:
# Build noise pool with matched first/second moments
X_t = torch.as_tensor(X, dtype=torch.float32)
mu = X_t.mean(0, keepdim=True)
std = X_t.std(0, unbiased=True, keepdim=True)
X_noise = mu + torch.randn_like(X_t) * std



In [ ]:
# Build sequence sets
experiments_prior = make_toy_experiment_list(stim_pool=list(name_to_idx.keys()), n_experiments=64, seed=0)
# For noise, create synthetic IDs and mapping
noise_names = [f'noise_{i}' for i in range(len(X_noise))]
noise_name_to_idx = {n: i for i, n in enumerate(noise_names)}
experiments_noise = make_toy_experiment_list(stim_pool=noise_names, n_experiments=64, seed=0)



In [ ]:
# Compare alpha=0 and alpha>0
for label, X0, n2i, exps in [
    ('prior-like', X_t, name_to_idx, experiments_prior),
    ('noise', X_noise, noise_name_to_idx, experiments_noise),
]:
    for a in [0.0, 0.05]:
        out = run_experiment_scores_prior(
            sigma0=sigma0_fitted, noise_mode=noise_mode, metric=metric,
            X0=X0, name_to_idx=n2i, experiment_list=exps,
            score_model=score_fn, drift_step_size=a,
        )
        auc = _compute_auroc_upper_envelope(out['hits'], out['fas'])
        print(label, a, auroc_to_dprime(auc))

